In [12]:
# Libraries
import pandas as pd
import numpy as np

# Load dataset
path="../../data/simon-data/properties-2b-located.csv"
df=pd.read_csv(path)
cols_to_clean=df[['price','number_of_rooms','living_area']]
cols_to_clean.isna().mean()

price              0.000000
number_of_rooms    0.016056
living_area        0.000000
dtype: float64

In [15]:
df

,url,zip_code,city,type_of_property,subtype_of_property,price,type_of_sale,number_of_rooms,living_area,fully_equipped_kitchen,...,open_fire,terrace,garden,garden_area,surface_of_the_land,number_of_facades,swimming_pool,state_of_the_building,province,region
0,https://immovlan.be/en/detail/residence/for-sa...,2200,Herentals,House,residence,369000,for sale,3.0,378,NaN,...,0,1,1,NaN,300.0,2.0,0,0.0,Antwerp,Flemish Region
1,https://immovlan.be/en/detail/apartment/for-sa...,1190,Vorst,Apartment,apartment,385000,for sale,2.0,100,NaN,...,0,1,0,0.0,NaN,NaN,0,NaN,Brussels,Brussels
2,https://immovlan.be/en/detail/residence/for-sa...,7100,La Louvière,House,residence,135000,for sale,3.0,125,1.0,...,0,0,1,NaN,NaN,2.0,0,NaN,Hainaut,Walloon Region
3,https://immovlan.be/en/detail/residence/for-sa...,2160,Wommelgem,House,residence,284000,for sale,3.0,180,NaN,...,0,0,1,240.0,295.0,2.0,0,0.0,Antwerp,Flemish Region
4,https://immovlan.be/en/detail/residence/for-sa...,6660,Houffalize,House,residence,299000,for sale,4.0,124,3.0,...,0,1,1,NaN,441.0,3.0,0,4.0,Luxembourg,Walloon Region
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12202,https://immovlan.be/en/detail/residence/for-sa...,9190,Stekene,House,residence,450000,for sale,3.0,170,NaN,...,0,0,1,NaN,300.0,3.0,0,4.0,East Flanders,Flemish Region
12203,https://immovlan.be/en/detail/apartment/for-sa...,1030,Schaarbeek,Apartment,apartment,229000,for sale,1.0,72,1.0,...,0,0,0,0.0,NaN,3.0,0,1.0,Brussels,Brussels
12204,https://immovlan.be/en/detail/residence/for-sa...,6470,Sautin,House,residence,199000,for sale,3.0,134,NaN,...,1,1,1,NaN,1497.0,NaN,0,NaN,Hainaut,Walloon Region
12205,https://immovlan.be/en/detail/residence/for-sa...,1200,Sint-Lambrechts-Woluwe,House,residence,1090000,for sale,5.0,243,2.0,...,0,1,1,NaN,237.0,2.0,0,1.0,Brussels,Brussels


In [14]:
# Get stats on missing values
print(f'Means:\n{cols_to_clean.mean()}\n')
print(f'Medians:\n{cols_to_clean.median()}\n')
print(f'Std:\n{cols_to_clean.std()}')

Means:
price              431239.494388
number_of_rooms         2.930064
living_area           164.652494
dtype: float64

Medians:
price              335000.0
number_of_rooms         3.0
living_area           133.0
dtype: float64

Std:
price              393017.081299
number_of_rooms         1.623958
living_area           179.528502
dtype: float64


In [ ]:
# Remove outliers (3*IQR = soft removing >< 1.5*IQR)
## Get quartiles
Q1=cols_to_clean.quantile(0.25)
print(Q1)
Q3=cols_to_clean.quantile(0.75)
print(Q3)
IQR=Q3-Q1
print(IQR)

## Get limits
low_limit=(Q1-3*IQR).clip(lower=0)
print(low_limit)
high_limit=Q3+3*IQR
print(high_limit)

### Find limit value for 'Garden area' for prop with garden
prop_with_garden=(df[df['garden_area']>0]==True).index
prop_with_garden=df.loc[prop_with_garden]
gardenQ1=prop_with_garden['garden_area'].quantile(0.25)
gardenQ3=prop_with_garden['garden_area'].quantile(0.75)
gardenIQR=gardenQ3-gardenQ1
garden_high_limit=gardenQ3+3*gardenIQR

### Change limit in high_limit
high_limit['garden_area']=garden_high_limit
high_limit

## Exclude outliers
### Create SubDataSet with replaced NaN values
sub_excl_manip=df
sub_excl_manip=sub_excl_manip.replace(np.nan,0)
cols_to_clean_names=cols_to_clean.columns
mask=sub_excl_manip[cols_to_clean_names].apply(lambda x: x.between(low_limit[x.name],high_limit[x.name])).all(axis=1)

### Retrieve Dataset without outliers
df=df[mask]


price              235000.0
number_of_rooms         2.0
living_area            90.0
Name: 0.25, dtype: float64
price              494000.0
number_of_rooms         4.0
living_area           193.0
Name: 0.75, dtype: float64
price              259000.0
number_of_rooms         2.0
living_area           103.0
dtype: float64
price              0.0
number_of_rooms    0.0
living_area        0.0
dtype: float64
price              1271000.0
number_of_rooms         10.0
living_area            502.0
dtype: float64


,url,zip_code,city,type_of_property,subtype_of_property,price,type_of_sale,number_of_rooms,living_area,fully_equipped_kitchen,...,open_fire,terrace,garden,garden_area,surface_of_the_land,number_of_facades,swimming_pool,state_of_the_building,province,region
0,https://immovlan.be/en/detail/residence/for-sa...,2200,Herentals,House,residence,369000,for sale,3.0,378,NaN,...,0,1,1,NaN,300.0,2.0,0,0.0,Antwerp,Flemish Region
1,https://immovlan.be/en/detail/apartment/for-sa...,1190,Vorst,Apartment,apartment,385000,for sale,2.0,100,NaN,...,0,1,0,0.0,NaN,NaN,0,NaN,Brussels,Brussels
2,https://immovlan.be/en/detail/residence/for-sa...,7100,La Louvière,House,residence,135000,for sale,3.0,125,1.0,...,0,0,1,NaN,NaN,2.0,0,NaN,Hainaut,Walloon Region
3,https://immovlan.be/en/detail/residence/for-sa...,2160,Wommelgem,House,residence,284000,for sale,3.0,180,NaN,...,0,0,1,240.0,295.0,2.0,0,0.0,Antwerp,Flemish Region
4,https://immovlan.be/en/detail/residence/for-sa...,6660,Houffalize,House,residence,299000,for sale,4.0,124,3.0,...,0,1,1,NaN,441.0,3.0,0,4.0,Luxembourg,Walloon Region
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12201,https://immovlan.be/en/detail/apartment/for-sa...,1090,Jette,Apartment,apartment,410000,for sale,4.0,197,NaN,...,0,1,0,0.0,NaN,3.0,0,0.0,Brussels,Brussels
12202,https://immovlan.be/en/detail/residence/for-sa...,9190,Stekene,House,residence,450000,for sale,3.0,170,NaN,...,0,0,1,NaN,300.0,3.0,0,4.0,East Flanders,Flemish Region
12203,https://immovlan.be/en/detail/apartment/for-sa...,1030,Schaarbeek,Apartment,apartment,229000,for sale,1.0,72,1.0,...,0,0,0,0.0,NaN,3.0,0,1.0,Brussels,Brussels
12204,https://immovlan.be/en/detail/residence/for-sa...,6470,Sautin,House,residence,199000,for sale,3.0,134,NaN,...,1,1,1,NaN,1497.0,NaN,0,NaN,Hainaut,Walloon Region


In [31]:
mask.value_counts()

True    11663
Name: count, dtype: int64

In [26]:
sub_excl_manip[cols_to_clean]

TypeError: Boolean array expected for the condition, not int64

In [23]:
mask

NameError: name 'mask' is not defined